In [ ]:
""" Import required libraries
"""
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
from pyproj import Transformer
import geopandas as gpd
from scipy.interpolate import griddata
import pandas as pd

In [ ]:
# profile coordinates
path_utig = datapath + 'BedMap3-radar-thk-profiles/bedmap3_raw_data/UTIG_2010_ICECAP_AIR_BM3.csv'
path_nasa = datapath + 'BedMap3-radar-thk-profiles/bedmap2_raw_data/NASA_2011_ICEBRIDGE_AIR_BM2.csv'

ds_utig = pd.read_csv(path_utig,skiprows=18,index_col=['trajectory_id'])
ds_nasa = pd.read_csv(path_nasa,skiprows=18,index_col=['trajectory_id'])

profile_A = 2011121303002
profile_B = 'IR1HI2_2010343_WSB_JKB1a_GL0099a'
profile_C = 'IR2HI2_2010343_WSB_JKB2c_GL0233b'
profile_D = 'IR1HI2_2010343_WSB_JKB1a_GL0367a'

transformer_profiles = Transformer.from_crs(
    "EPSG:4326",   # lon/lat (WGS84)
    "EPSG:3031",   # Antarctic Polar Stereographic
    always_xy=True
)

XA,YA = transformer_profiles.transform(ds_nasa.loc[profile_A]['longitude (degree_east)'], ds_nasa.loc[profile_A]['latitude (degree_north)'])
XB,YB = transformer_profiles.transform(ds_utig.loc[profile_B]['longitude (degree_east)'], ds_utig.loc[profile_B]['latitude (degree_north)'])
XC,YC = transformer_profiles.transform(ds_utig.loc[profile_C]['longitude (degree_east)'], ds_utig.loc[profile_C]['latitude (degree_north)'])
XD,YD = transformer_profiles.transform(ds_utig.loc[profile_D]['longitude (degree_east)'], ds_utig.loc[profile_D]['latitude (degree_north)'])

# xA =  398000
# yA = -942000
# xAA = 410000
# yAA = -950000

# xB = 410000
# yB = -910000
# xBB = 440000
# yBB = -930000

# xC = 435000.
# yC = -845000.
# xCC = 480000.
# yCC = -870000.

# xD = 410000
# yD = -875000
# xDD = 450000.
# yDD = -900000.

data = sio.loadmat(f'{homepath}/saved-data/exp1-ice-thickness-BedMap-grid.mat')
x = data['x']
y = data['y']
thk = data['median_thickness']

fig,ax = plt.subplots(1,1)
ax1 = ax.pcolormesh(x,y,thk.reshape((x.shape[0],x.shape[1])))
ax.set_aspect(1)
# ax.plot([xA,xAA],[yA,yAA],color='red')
# ax.plot([xB,xBB],[yB,yBB],color='red')
# ax.plot([xC,xCC],[yC,yCC],color='red')
# ax.plot([xD,xDD],[yD,yDD],color='red')

ax.scatter(XA[10:650],YA[10:650],s=0.1)
ax.scatter(XB[15700:17600],YB[15700:17600],s=0.1)
ax.scatter(XC[1600:3500],YC[1600:3500],s=0.1)
ax.scatter(XD[12000:14700],YD[12000:14700],s=0.1)

In [ ]:
XA = XA[10:650].reshape((len(XA[10:650]),1))
YA = YA[10:650].reshape((len(YA[10:650]),1))
XB = XB[15700:17600].reshape((len(XB[15700:17600]),1))
YB = YB[15700:17600].reshape((len(YB[15700:17600]),1))
XC = XC[1600:3500].reshape((len(XC[1600:3500]),1))
YC = YC[1600:3500].reshape((len(YC[1600:3500]),1))
XD = XD[12000:14700].reshape((len(XD[12000:14700]),1))
YD = YD[12000:14700].reshape((len(YD[12000:14700]),1))

bm3_p1 = griddata(XYs,bedcrop.flatten(),(XA,YA),method='linear')
bmap3_p1 = griddata(XYb2,bed2crop.flatten(),(XA,YA),method='linear')
s_p1 = griddata(XYb2,scrop.flatten(),(XA,YA),method='linear')
s2_p1 = griddata(XYs,s2crop.flatten(),(XA,YA),method='linear')

bm3_p2 = griddata(XYs,bedcrop.flatten(),(XB,YB),method='linear')
bmap3_p2 = griddata(XYb2,bed2crop.flatten(),(XB,YB),method='linear')
s_p2 = griddata(XYb2,scrop.flatten(),(XB,YB),method='linear')
s2_p2 = griddata(XYs,s2crop.flatten(),(XB,YB),method='linear')

bm3_p3 = griddata(XYs,bedcrop.flatten(),(XC,YC),method='linear')
bmap3_p3 = griddata(XYb2,bed2crop.flatten(),(XC,YC),method='linear')
s_p3 = griddata(XYb2,scrop.flatten(),(XC,YC),method='linear')
s2_p3 = griddata(XYs,s2crop.flatten(),(XC,YC),method='linear')

bm3_p4 = griddata(XYs,bedcrop.flatten(),(XD,YD),method='linear')
bmap3_p4 = griddata(XYb2,bed2crop.flatten(),(XD,YD),method='linear')
s_p4 = griddata(XYb2,scrop.flatten(),(XD,YD),method='linear')
s2_p4 = griddata(XYs,s2crop.flatten(),(XD,YD),method='linear')

Hobs = [[bm3_p1,bmap3_p1,s_p1,s2_p1],
        [bm3_p2,bmap3_p2,s_p2,s2_p2],
        [bm3_p3,bmap3_p3,s_p3,s2_p3],
        [bm3_p4,bmap3_p4,s_p4,s2_p4],]

for i,name in zip(np.arange(2),['bm3','bmap3']):
    for j,x,y in zip(np.arange(4),[XA,XB,XC,XD],[YA,YB,YC,YD]):
        dataH = {'x':x,'y':y,'interpolated_thickness':Hobs[j][i]}
        sio.savemat(f'{homepath}/saved-data/profile-p{j+1}-{name}-interpolated-thickness.mat',dataH)

for i,name in zip(np.arange(2),['bmap3','bm3']):
    for j,x,y in zip(np.arange(4),[XA,XB,XC,XD],[YA,YB,YC,YD]):
        dataH = {'x':x,'y':y,'interpolated_surface':Hobs[j][i+2]}
        sio.savemat(f'{homepath}/saved-data/profile-p{j+1}-{name}-interpolated-surface.mat',dataH)


In [ ]:
# comparison profiles
# prepare data
# runtime ca. 100 min

data1 = sio.loadmat(f'{homepath}/saved-data/exp1-ice-thickness-BedMap-grid.mat')
data2 = sio.loadmat(f'{homepath}/saved-data/exp5.1-ice-thickness-BedMap-grid.mat')
data3 = sio.loadmat(f'{homepath}/saved-data/exp5.6-ice-thickness-BedMap-grid.mat')

data4 = sio.loadmat(f'{homepath}/saved-data/exp4.2-ice-thickness-ItsLive-grid.mat')
data5 = sio.loadmat(f'{homepath}/saved-data/exp4.3-ice-thickness-ItsLive-grid.mat')
data6 = sio.loadmat(f'{homepath}/saved-data/exp5.2-ice-thickness-ItsLive-grid.mat')

xBM = data1['x']
yBM = data1['y']
xIL = data4['x']
yIL = data4['y']

XBM = xBM.flatten()
YBM = yBM.flatten()
XBM = XBM.reshape((XBM.size,1))
YBM = YBM.reshape((YBM.size,1))
XYBM = np.hstack((XBM,YBM))

XIL = xIL.flatten()
YIL = yIL.flatten()
XIL = XIL.reshape((XIL.size,1))
YIL = YIL.reshape((YIL.size,1))
XYIL = np.hstack((XIL,YIL))

H1 = data1['mean_thickness']
H2 = data2['mean_thickness']
H3 = data3['mean_thickness']
H4 = data4['mean_thickness']
H5 = data5['mean_thickness']
H6 = data6['mean_thickness']


XA = np.linspace(xA,xAA,50)
YA = np.linspace(yA,yAA,50)
XA = XA.reshape((50,1))
YA = YA.reshape((50,1))
# xy = np.hstack((x,y))

XB = np.linspace(xB,xBB,50)
YB = np.linspace(yB,yBB,50)
XB = XB.reshape((50,1))
YB = YB.reshape((50,1))

XC = np.linspace(xC,xCC,50)
YC = np.linspace(yC,yCC,50)
XC = XC.reshape((50,1))
YC = YC.reshape((50,1))

XD = np.linspace(xD,xDD,50)
YD = np.linspace(yD,yDD,50)
XD = XD.reshape((50,1))
YD = YD.reshape((50,1))

# interpolate data to profiles
# for x,y in zip([XA],[YA]):
bm3_p1 = griddata(XYs,bedcrop.flatten(),(XA,YA),method='linear')
bmap3_p1 = griddata(XYb2,bed2crop.flatten(),(XA,YA),method='linear')
s_p1 = griddata(XYb2,scrop.flatten(),(XA,YA),method='linear')

H1_p1 = griddata(XYBM,H1.flatten(),(XA,YA),method='linear')
H2_p1 = griddata(XYBM,H2.flatten(),(XA,YA),method='linear')
H3_p1 = griddata(XYBM,H3.flatten(),(XA,YA),method='linear')
H4_p1 = griddata(XYIL,H4.flatten(),(XA,YA),method='linear')
H5_p1 = griddata(XYIL,H5.flatten(),(XA,YA),method='linear')
H6_p1 = griddata(XYIL,H6.flatten(),(XA,YA),method='linear')

bm3_p2 = griddata(XYs,bedcrop.flatten(),(XB,YB),method='linear')
bmap3_p2 = griddata(XYb2,bed2crop.flatten(),(XB,YB),method='linear')
s_p2 = griddata(XYb2,scrop.flatten(),(XB,YB),method='linear')

H1_p2 = griddata(XYBM,H1.flatten(),(XB,YB),method='linear')
H2_p2 = griddata(XYBM,H2.flatten(),(XB,YB),method='linear')
H3_p2 = griddata(XYBM,H3.flatten(),(XB,YB),method='linear')
H4_p2 = griddata(XYIL,H4.flatten(),(XB,YB),method='linear')
H5_p2 = griddata(XYIL,H5.flatten(),(XB,YB),method='linear')
H6_p2 = griddata(XYIL,H6.flatten(),(XB,YB),method='linear')

bm3_p3 = griddata(XYs,bedcrop.flatten(),(XC,YC),method='linear')
bmap3_p3 = griddata(XYb2,bed2crop.flatten(),(XC,YC),method='linear')
s_p3 = griddata(XYb2,scrop.flatten(),(XC,YC),method='linear')

H1_p3 = griddata(XYBM,H1.flatten(),(XC,YC),method='linear')
H2_p3 = griddata(XYBM,H2.flatten(),(XC,YC),method='linear')
H3_p3 = griddata(XYBM,H3.flatten(),(XC,YC),method='linear')
H4_p3 = griddata(XYIL,H4.flatten(),(XC,YC),method='linear')
H5_p3 = griddata(XYIL,H5.flatten(),(XC,YC),method='linear')
H6_p3 = griddata(XYIL,H6.flatten(),(XC,YC),method='linear')

bm3_p4 = griddata(XYs,bedcrop.flatten(),(XD,YD),method='linear')
bmap3_p4 = griddata(XYb2,bed2crop.flatten(),(XD,YD),method='linear')
s_p4 = griddata(XYb2,scrop.flatten(),(XD,YD),method='linear')

H1_p4 = griddata(XYBM,H1.flatten(),(XD,YD),method='linear')
H2_p4 = griddata(XYBM,H2.flatten(),(XD,YD),method='linear')
H3_p4 = griddata(XYBM,H3.flatten(),(XD,YD),method='linear')
H4_p4 = griddata(XYIL,H4.flatten(),(XD,YD),method='linear')
H5_p4 = griddata(XYIL,H5.flatten(),(XD,YD),method='linear')
H6_p4 = griddata(XYIL,H6.flatten(),(XD,YD),method='linear')


In [ ]:
XA = np.linspace(xA,xAA,50)
YA = np.linspace(yA,yAA,50)
XA = XA.reshape((50,1))
YA = YA.reshape((50,1))
# xy = np.hstack((x,y))

XB = np.linspace(xB,xBB,50)
YB = np.linspace(yB,yBB,50)
XB = XB.reshape((50,1))
YB = YB.reshape((50,1))

XC = np.linspace(xC,xCC,50)
YC = np.linspace(yC,yCC,50)
XC = XC.reshape((50,1))
YC = YC.reshape((50,1))

XD = np.linspace(xD,xDD,50)
YD = np.linspace(yD,yDD,50)
XD = XD.reshape((50,1))
YD = YD.reshape((50,1))

s2_p1 = griddata(XYs,s2crop.flatten(),(XA,YA),method='linear')
s2_p2 = griddata(XYs,s2crop.flatten(),(XB,YB),method='linear')
s2_p3 = griddata(XYs,s2crop.flatten(),(XC,YC),method='linear')
s2_p4 = griddata(XYs,s2crop.flatten(),(XD,YD),method='linear')

In [ ]:
H_p = [[H1_p1,H2_p1,H3_p1,H4_p1,H5_p1,H6_p1],
       [H1_p2,H2_p2,H3_p2,H4_p2,H5_p2,H6_p2],
       [H1_p3,H2_p3,H3_p3,H4_p3,H5_p3,H6_p3],
       [H1_p4,H2_p4,H3_p4,H4_p4,H5_p4,H6_p4]]

In [ ]:
Hobs = [[bm3_p1,bmap3_p1,s_p1],
        [bm3_p2,bmap3_p2,s_p2],
        [bm3_p3,bmap3_p3,s_p3],
        [bm3_p4,bmap3_p4,s_p4],]

In [ ]:
s2obs = [s2_p1,
        s2_p2,
        s2_p3,
        s2_p4]

In [ ]:
for i in np.arange(6):
    for j,x,y in zip(np.arange(4),[XA,XB,XC,XD],[YA,YB,YC,YD]):
        dataH = {'x':x,'y':y,'interpolated_thickness':H_p[j][i]}
        sio.savemat(f'{homepath}/saved-data/H{i+1}_p{j+1}-interpolated-thickness.mat',dataH)

In [ ]:
for i,name in zip(np.arange(2),['bm3','bmap3']):
    for j,x,y in zip(np.arange(4),[XA,XB,XC,XD],[YA,YB,YC,YD]):
        dataH = {'x':x,'y':y,'interpolated_thickness':Hobs[j][i]}
        sio.savemat(f'{homepath}/saved-data/{name}_p{j+1}-interpolated-thickness.mat',dataH)

In [ ]:
for j,x,y in zip(np.arange(4),[XA,XB,XC,XD],[YA,YB,YC,YD]):
    dataH = {'x':x,'y':y,'interpolated_surface':Hobs[j][2]}
    sio.savemat(f'{homepath}/saved-data/bm3_p{j+1}-interpolated-surface.mat',dataH)

In [ ]:
for j,x,y in zip(np.arange(4),[XA,XB,XC,XD],[YA,YB,YC,YD]):
    dataH = {'x':x,'y':y,'interpolated_surface':s2obs[j]}
    sio.savemat(f'{homepath}/saved-data/bm3_p{j+1}-interpolated-surface.mat',dataH)